# 6-4 zero_grad, backward, step 순서 — 심화

직접 작성한 코드와 저장된 실행 결과를 정리했습니다. 첫 문제만 실행되어 나머지 셀은 주말 재실행 대상으로 남겼습니다.


In [1]:
# 검증 가능 정답 코드
# 새 loss 그래프를 만들더라도 같은 leaf의 .grad 저장소는 자동으로 비워지지 않는 상황을 두 번 backward해 재현합니다.
import torch

w = torch.tensor(1.0, requires_grad=True)

loss1 = (2 * w) ** 2
loss1.backward()
first = w.grad.item()

# 그래프는 새로 계산하지만 .grad를 비우지 않아 값은 누적됩니다.
loss2 = (2 * w) ** 2
loss2.backward()
accumulated = w.grad.item()

# 다음 독립 step을 가정해 이전 gradient를 지웁니다.
w.grad.zero_()
loss3 = (2 * w) ** 2
loss3.backward()
reset = w.grad.item()

# 명시적 zeroing 뒤 세 번째 gradient가 첫 값 8로 돌아오는지 확인해 입력 변화가 아닌 누적임을 증명합니다.
print(f"first_grad={first:.1f}")
print(f"accumulated_grad={accumulated:.1f}")
print(f"reset_grad={reset:.1f}")

first_grad=8.0
accumulated_grad=16.0
reset_grad=8.0


In [ ]:
# 검증 가능 정답 코드
# 단일 step은 zeroing→forward→scalar loss→backward→step 순서로 완결해 batch 사이 gradient 경계를 고정합니다.
import torch
import torch.nn as nn

model = nn.Linear(1, 1, bias=False)
with torch.no_grad():
    model.weight.fill_(1.0)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
criterion = nn.MSELoss()
x = torch.tensor([[1.0], [2.0]])
y = torch.zeros_like(x)

def train_step(model, optimizer, criterion, x, y):
    model.train()
    # 반드시 현재 배치의 forward보다 앞에서 이전 gradient를 제거합니다.
    optimizer.zero_grad()
    pred = model(x)
    loss = criterion(pred, y)
    loss.backward()
    optimizer.step()
    return loss.item()

loss_value = train_step(model, optimizer, criterion, x, y)
# 고정 입력의 손계산 loss 2.5와 예상 weight 0.5를 실제 출력과 대조해 optimizer 연결까지 검산합니다.
print(f"loss={loss_value:.1f}")
print(f"weight_after={model.weight.item():.1f}")

In [ ]:
# 검증 가능 정답 코드
# 두 후보를 같은 초기 weight와 같은 두 batch에서 실행하고 두 번째 step 전 zeroing 여부만 다르게 둡니다.
import torch
import torch.nn as nn

x = torch.tensor([[1.0], [2.0]])
y = torch.zeros_like(x)
criterion = nn.MSELoss()

def run(clear_each_step):
    model = nn.Linear(1, 1, bias=False)
    with torch.no_grad():
        model.weight.fill_(1.0)
    opt = torch.optim.SGD(model.parameters(), lr=0.1)
    for _ in range(2):
        if clear_each_step:
            opt.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        opt.step()
    return model.weight.item()

weight_c = run(True)
weight_d = run(False)
# 정상 0.25와 누적 -0.25를 함께 출력해 우연한 loss 순위가 아니라 계약을 지킨 C를 승인합니다.
print(f"correct_weight={weight_c:.2f}")
print(f"accumulated_weight={weight_d:.2f}")
print("approved=C")